## Visão Geral do Microsoft Entra ID

Microsoft Entra ID (anteriormente Azure Active Directory) é o serviço de gerenciamento de identidade e acesso baseado em nuvem da Microsoft. Ele serve como provedor de identidade central para Microsoft 365, Azure e milhares de outras aplicações SaaS.

Recursos Principais:
* **Single Sign-On (SSO)** - Usuários se autenticam uma vez para acessar múltiplas aplicações
* **Multi-Factor Authentication (MFA)** - Segurança aprimorada através de métodos adicionais de verificação
* **Conditional Access** - Controle de acesso baseado em políticas com base em usuário, dispositivo, localização e risco
* **Application Integration** - Suporta protocolos de autenticação modernos como OAuth 2.0, OpenID Connect e SAML

## Visão Geral do Amazon Bedrock Gateway

Bedrock AgentCore Gateway fornece aos clientes uma maneira de transformar suas APIs existentes e funções Lambda em servidores MCP totalmente gerenciados sem precisar gerenciar infraestrutura ou hospedagem. Os clientes podem trazer especificações OpenAPI ou modelos Smithy para suas APIs existentes, ou adicionar funções Lambda que estão à frente de suas ferramentas. O Gateway fornecerá uma interface Model Context Protocol (MCP) uniforme em todas essas ferramentas. O Gateway emprega um modelo de autenticação dupla para garantir controle de acesso seguro tanto para requisições de entrada quanto para conexões de saída para recursos de destino. A estrutura consiste em dois componentes principais: Inbound Auth, que valida e autoriza usuários tentando acessar alvos do gateway, e Outbound Auth, que permite ao gateway conectar-se com segurança a recursos backend em nome de usuários autenticados. Juntos, esses mecanismos de autenticação criam uma ponte segura entre usuários e seus recursos de destino, suportando tanto credenciais IAM quanto fluxos de autenticação baseados em OAuth. O Gateway suporta conexão de transporte HTTP Streamable do MCP.

Mais detalhes sobre Amazon Bedrock AgentCore Gateway podem ser encontrados em:
- https://github.com/awslabs/amazon-bedrock-agentcore-samples/tree/main/01-tutorials/02-AgentCore-gateway
- https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway.html

## Objetivo de Aprendizado
Microsoft EntraID pode ser usado como provedor de identidade no AgentCore Identity para autorizar o acesso da aplicação consumidora a recursos protegidos do Amazon AgentCore Gateway. Neste notebook exploraremos o uso do EntraID para autenticação de entrada com Amazon Bedrock Gateway.

## Objetivo de Aprendizado 1: Configurar Entra ID para uso com AgentCore Gateway

### Passo 1: Configurar Tenant do Entra ID
Um tenant do Entra ID é uma instância dedicada do Microsoft Entra ID que representa sua organização. Pense nele como o diretório isolado da sua organização na nuvem da Microsoft.

Características Principais:
* **Identidade Única** - Cada tenant tem um domínio único (por exemplo, suaempresa.onmicrosoft.com)
* **Limite Isolado** - Usuários, grupos e aplicações em um tenant são separados de outros
* **Controle Administrativo** - Administradores do tenant gerenciam usuários, políticas de segurança e registros de aplicação
* **Suporte Multi-Domínio** - Pode incluir domínios personalizados além do domínio padrão .onmicrosoft.com

Na Prática:
Quando você registra uma aplicação com o Entra ID para integração OAuth, você está registrando-a dentro de um tenant específico. Usuários desse tenant podem então se autenticar em sua aplicação usando suas credenciais organizacionais.

Para integração com AgentCore, você precisará de:
* **Tenant ID** - Identificador único para a instância do Entra ID
* **Application Registration** - Sua aplicação registrada dentro do tenant
* **Permissões Apropriadas** - Direitos de acesso configurados para sua aplicação

Este modelo baseado em tenant garante que autenticação e autorização permaneçam dentro do limite de segurança da sua organização.

Passos para criar um tenant podem ser encontrados em https://learn.microsoft.com/en-us/entra/fundamentals/create-new-tenant

Nota:
1. MS EntraID não é um serviço AWS. Consulte a documentação do Microsoft EntraID para custos relacionados ao EntraID.
2. Capturas de tela usadas nos seguintes passos podem mudar. Encorajamos você a consultar a documentação do Microsoft Entra ID para orientações mais recentes sobre configuração de aplicação EntraID.

In [ ]:
import os
os.environ["tenant_id"] = "REPLACE_ME" # Replace with Tenant ID from EntraID

### Passo 2: Definir a API que você deseja usar
1. Vá para portal.azure.com e pesquise por "Entra ID" na barra de pesquisa no topo da tela
<img src="images/entraid.jpg" width="75%">
2. Vá para manage --> App Registrations
<img src="images/app.registration.png" width="75%">
3. Clique em "New Registration" e preencha os detalhes. Selecione a opção multi tenant
- Não defina nenhuma URL de redirecionamento.
<img src="images/setup.api.png" width="75%">
4. Exponha uma API através de Manage --> Expose an API
<img src="images/api.expose.png" width="75%"/>
5. Crie funções de aplicativo para a API. Não estamos adicionando scopes já que esta é uma configuração M2M.
<img src="images/weather.app.role.png" width="75%"/>

In [ ]:
# app_id_url will be "Application ID URI" from "App registration" --> "All Applications" --> Select client you just created --> "Expose a API".
os.environ["app_id_uri"] = "api://3dXXXXXX-CCCC-VVVV-BBBB-NNNNNN885f25" # This is the API URL you set up for "weather_service"

### Passo 3: Criar uma aplicação Cliente Entra
1. Vá para portal.azure.com e pesquise por "Entra ID" na barra de pesquisa no topo da tela

<img src="images/entraid.jpg" width="75%">

2. Vá para manage --> App Registrations

<img src="images/app.registration.png" width="75%">

3. Clique em "New Registration" e preencha os detalhes. Selecione a opção multi tenant
- Não defina nenhuma URL de redirecionamento.

<img src="images/client.register.png" width="75%"/>

4. Crie um client secret. Copie o client secret e client ID para uso no AgentCore.

<img src="images/client.secret.png" width="75%">

5. Vá para API permissions e solicite permissões para a API que você criou anteriormente. "API Permissions" --> "Add a Permission" --> "APIs my organization uses" e pesquise pela API que você criou no passo 1

<img src="images/api.permissions.png" width="75%">

6. Conceda consentimento de administrador para usar as APIs.

7. Configure variáveis de ambiente usando as informações do EntraID

In [ ]:
import os
# Tenant ID from "App registration" --> "All Applications" --> Select client you just created --> "Overview" --> "Directory (tenant) ID"
os.environ["tenant_id"] = "bc24XXXX-CCCC-VVVV-BBBB-NNNNb5df1f19"

# Client ID from "App registration" --> "All Applications" --> Select client you just created --> "Overview" --> "Application (client) ID"
os.environ["client_id"] = "08XXXXXX-CCCC-VVVV-BBBB-NNNNNNd86cd2" # Replace with Client ID of the "weather_service_client"

# Secret saved from earlier step
os.environ["client_secret"] = "muCCCCCVVVVVBBBBBNNNNN3dY6qdlL" # Replace with Client secret of the "weather_service_client"

## Objetivo de Aprendizado 2: Configurar AgentCore Gateway e Lambda Target

### Passo 1: Criar uma Função Lambda para usar com Entra ID
1. Crie um arquivo python que usaremos como código da função lambda. Note como o nome da ferramenta sendo chamada é recuperado do objeto `context` e usado na função lambda.

In [ ]:
import boto3
import zipfile
import io
from botocore.exceptions import ClientError
from boto3.session import Session
import time
import uuid

boto_session = Session()
sts = boto3.client('sts')
region = boto_session.region_name
account_id = sts.get_caller_identity().get("Account")

In [ ]:
%%writefile lambda_function.py
def lambda_handler(event, context):
    print(f"Event: {event}")
    print(f"Context: {context}")
    extended_tool_name = context.client_context.custom["bedrockAgentCoreToolName"]
    resource = extended_tool_name.split("___")[1]

    print(resource)
    city = event.get("city")
    print(city)
    if resource == "weather_check":
        return f"Weather in {city} is bright and sunny!"
    elif resource == "directions":
        return f"Take I5 south all the way to {city} downtown"

2. Criar a função lambda

In [ ]:
lambda_client = boto3.client('lambda', region_name=region)
with zipfile.ZipFile('lambda_function.zip', 'w') as zip_file:
    zip_file.write('lambda_function.py', 'lambda_function.py')

with open('lambda_function.zip', 'rb') as zip_file:
    zip_content = zip_file.read()

In [ ]:
iam_client = boto3.client('iam', region_name=region)

trust_policy = """{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {
                "Service": "lambda.amazonaws.com"
            },
            "Action": "sts:AssumeRole"
        }
    ]
}
"""

policy = """{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "logs:CreateLogGroup",
                "logs:CreateLogStream",
                "logs:PutLogEvents"
            ],
            "Resource": "arn:aws:logs:*:*:*"
        }
    ]
}
"""

response = iam_client.create_role(
    RoleName='lambda-role',
    AssumeRolePolicyDocument=trust_policy
)

iam_client.put_role_policy(
        PolicyDocument=policy,
        PolicyName="lambda-policy",
        RoleName="lambda-role"
    )

lambda_role_arn = response['Role']['Arn']

# Wait for role to propagate
time.sleep(10)

response = lambda_client.create_function(
    FunctionName='m2m-entra-lambda',
    Runtime='python3.12',
    Role=lambda_role_arn,
    Handler='lambda_function.lambda_handler',
    Code={'ZipFile': zip_content},
)

In [ ]:
lambda_arn = response["FunctionArn"]

In [ ]:
lambda_arn

### Passo 2: Criar Amazon Bedrock AgentCore Gateway com segurança de entrada

In [ ]:
iam_client = boto3.client('iam')

trust_policy = """{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {
                "Service": "bedrock-agentcore.amazonaws.com"
            },
            "Action": "sts:AssumeRole"
        }
    ]
}
"""

# Create role with trust policy
response = iam_client.create_role(
    RoleName='bedrock-agent-lambda-role',
    AssumeRolePolicyDocument=trust_policy
)

permission = """{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Action": [
                "lambda:InvokeFunction"
            ],
            "Resource": [
                "%s"
            ],
            "Effect": "Allow",
            "Sid": "InvokeFunction"
        }
    ]
}
"""% lambda_arn


# Add Lambda invoke policy
iam_client.put_role_policy(
    RoleName='bedrock-agent-lambda-role',
    PolicyName='lambda-invoke-policy',
    PolicyDocument=permission
)

role_arn = response['Role']['Arn']
print(f"Role ARN: {role_arn}")

In [ ]:
gateway_client = boto3.client(
    "bedrock-agentcore-control",
    region_name=region,
)

gateway_name = "m2m-entra-gateway"
auth_config = {
    "customJWTAuthorizer": {
        "allowedAudience": [
            os.environ["app_id_uri"]
        ],
        "discoveryUrl": f"https://login.microsoftonline.com/{os.environ["tenant_id"]}/.well-known/openid-configuration"
    }
}

In [ ]:
create_response = gateway_client.create_gateway(
    name=gateway_name,
    roleArn= role_arn,
    protocolType="MCP",
    authorizerType="CUSTOM_JWT",
    authorizerConfiguration=auth_config,
    description="Customer Support AgentCore Gateway",
)

In [ ]:
gateway_url = create_response["gatewayUrl"]
gateway_id = create_response["gatewayId"]

### Passo 3: Adicionar alvo lambda ao AgentCore Gateway que acabamos de criar

1. Especificação de API para as ferramentas reais que estamos criando através da função lambda.

In [ ]:
api_spec = [
    {
        "name": "weather_check",
        "description": "Check the weather for a given City",
        "inputSchema": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "The city you want to get weather for"
                }
            },
            "required": [
                "city"
            ]
        }
    },
    {
        "name": "directions",
        "description": "Search the web for directions to a city",
        "inputSchema": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "The city you want to get directions to"
                }
            },
            "required": [
                "city"
            ]
        }
    }
]

In [ ]:
lambda_target_config = {
    "mcp": {
        "lambda": {
            "lambdaArn": lambda_arn,
            "toolSchema": {"inlinePayload": api_spec},
        }
    }
}

# Create gateway target
credential_config = [{"credentialProviderType": "GATEWAY_IAM_ROLE"}]

create_target_response = gateway_client.create_gateway_target(
    gatewayIdentifier=gateway_id,
    name="LambdaUsingSDK",
    description="Lambda Target using SDK",
    targetConfiguration=lambda_target_config,
    credentialProviderConfigurations=credential_config,
)

## Objetivo de Aprendizado 3: Usar as ferramentas disponibilizadas através do AgentCore Gateway no seu agente

### Passo 1: Obter um token e revisar o payload e cabeçalho
1. Obtenha um token de acesso e use-o para acessar o AgentCore Gateway.

In [ ]:
import requests
import json

TOKEN_URL = f"https://login.microsoftonline.com/{os.environ["tenant_id"]}/oauth2/v2.0/token"
SCOPE=f"{os.environ["app_id_uri"]}/.default"

def fetch_access_token(client_id, client_secret, token_url,scope):

  data = {
      "grant_type":"client_credentials",
      "client_id":client_id,
      "client_secret": client_secret,
      "scope":scope
  }
    
  response = requests.post(
    token_url,
    data=data,
    headers={'Content-Type': 'application/x-www-form-urlencoded'}
  )
  #print(response.text)
  return response.json()['access_token']

access_token = fetch_access_token(os.environ["client_id"], os.environ["client_secret"], TOKEN_URL, SCOPE)

2. Decodifique-o e veja os conteúdos. Certifique-se de que "aud", "appid" e "roles" correspondem ao que você configurou anteriormente.

In [ ]:
import base64
import json

def decode_jwt_token(token):
    # Split the JWT into parts
    parts = token.split('.')
    
    # Decode header
    header = json.loads(base64.b64decode(parts[0] + '==').decode('utf-8'))
    
    # Decode payload
    payload = json.loads(base64.b64decode(parts[1] + '==').decode('utf-8'))
    
    return header, payload

# Usage
header, payload = decode_jwt_token(access_token)

print("Header:", json.dumps(header, indent=2))
print("Payload:", json.dumps(payload, indent=2))

# Check specific claims
print(f"Audience: {payload.get('aud')}")
print(f"Issuer: {payload.get('iss')}")
print(f"Expires: {payload.get('exp')}")
print(f"Scopes: {payload.get('scp')}")
print(f"Roles: {payload.get('roles')}")

### Passo 2: Use o token de acesso para obter a lista de ferramentas disponíveis do AgentCore Gateway
Você deve ver uma especificação de ferramenta similar à abaixo.

<img src="images/tools.spec.png" width="50%"/>

In [ ]:
def list_tools(gateway_url, access_token):
  headers = {
      "Content-Type": "application/json",
      "Authorization": f"Bearer {access_token}"
  }

  payload = {
      "jsonrpc": "2.0",
      "id": "list-tools-request",
      "method": "tools/list"
  }

  response = requests.post(gateway_url, headers=headers, json=payload)
  return response.json()
tools = list_tools(gateway_url, access_token)
print(json.dumps(tools, indent=2))

### Passo 3: Criar um cliente mcp, obter lista de ferramentas e usá-la em um agente Strands.

In [ ]:
from mcp.client.streamable_http import streamablehttp_client
from strands.tools.mcp import MCPClient

# Set up MCP client
mcp_client = MCPClient(
    lambda: streamablehttp_client(
        gateway_url,
        headers={"Authorization": f"Bearer {access_token}"},
    )
)

In [ ]:
mcp_client.start()

In [ ]:
mcp_client.list_tools_sync()

In [ ]:
from strands import Agent
agent = Agent(tools=mcp_client.list_tools_sync())

#### Nota: A resposta da função Lambda definida anteriormente é estática. Como resultado, a resposta deste agente será muito similar independentemente da cidade que você nomear no prompt.

In [ ]:
agent("What is the weather in San Diego?")

In [ ]:
agent("Give me directions to San Diego?")

## Conclusão e Limpeza
Neste notebook aprendemos como:
- Configurar API e Aplicação do Entra ID para fornecer fluxo OAuth Client Credential (M2M)
- Criar um AgentCore Gateway
- Criar uma função lambda e adicioná-la como alvo no AgentCore Gateway que criamos. Funções Lambda estarão disponíveis como ferramentas MCP através do AgentCore Gateway.
- Usar cliente MCP para acessar ferramentas fornecidas através do Gateway, vincular as ferramentas a um Agente Strands, e usá-lo para atender consultas do usuário.

#### Recursos criados

In [ ]:
lambda_arn, role_arn, gateway_id, lambda_role_arn, create_response["gatewayArn"]

In [ ]:
create_target_response["targetId"]

#### Deletar alvo lambda no seu Gateway.

In [ ]:
gateway_client.delete_gateway_target(gatewayIdentifier=gateway_id, targetId=create_target_response["targetId"])

#### Deletar gateway

In [ ]:
gateway_client.delete_gateway(gatewayIdentifier=gateway_id)

#### Deletar função lambda que você criou.

In [ ]:
function_name = lambda_arn.split(':')[-1]
lambda_client.delete_function(FunctionName=function_name)

#### Deletar roles criadas

In [ ]:
role_name = lambda_role_arn.split('/')[-1]
inline = iam_client.list_role_policies(RoleName=role_name)
for policy_name in inline['PolicyNames']:
    iam_client.delete_role_policy(RoleName=role_name, PolicyName=policy_name)
iam_client.delete_role(RoleName=role_name)

In [ ]:
role_name = role_arn.split('/')[-1]
inline = iam_client.list_role_policies(RoleName=role_name)
for policy_name in inline['PolicyNames']:
    iam_client.delete_role_policy(RoleName=role_name, PolicyName=policy_name)
iam_client.delete_role(RoleName=role_name)